In [1]:
from __future__ import annotations

import pathlib
import warnings
import dotenv

import lightning as L
import pandas as pd
import yaml
from transformertf.data import EncoderDecoderDataModule, MinMaxScaler

from sa_preisach.callbacks import PlotHysteresisCallback
from sa_preisach.models import EncoderDecoderPreisachNN
from sa_preisach.nn import PreisachLSTMEncoder, PreisachGRUEncoder
from sa_preisach.priors import DiagonalDensityPrior, CompositeDensityPrior, SymmetryDensityPrior, CentroidDensityPrior
from sa_preisach.utils import DefaultMeshSizeFunction

dotenv.load_dotenv()

L.pytorch.utilities.disable_possible_user_warnings()

warnings.filterwarnings(
    "ignore",
    message=".*'pin_memory' argument is set as true but not supported on MPS now.*",
    category=UserWarning
)

# Ignore the LeafSpec deprecation
warnings.filterwarnings(
    "ignore",
    message=r".*`isinstance\(treespec, LeafSpec\)` is deprecated.*",
    category=DeprecationWarning
)

# Ignore the num_workers bottleneck warnings for train and val loaders
warnings.filterwarnings(
    "ignore",
    message=r".*does not have many workers which may be a bottleneck.*",
    category=UserWarning
)

warnings.filterwarnings(
    "ignore",
    message=r".*`isinstance\(treespec, LeafSpec\)` is deprecated.*",
    category=FutureWarning
)


In [2]:
DATA_DIR = pathlib.Path("data")

DATA_MANIFEST_PATH = DATA_DIR / "gsi_armco_manifest.yaml"

with open(DATA_MANIFEST_PATH, encoding="utf-8") as f:
    data_manifest = yaml.safe_load(f)

data_manifest

{'initial_mag': ['data/GSI_ARMCO/initial_mag/spec_1.parquet',
  'data/GSI_ARMCO/initial_mag/spec_2.parquet',
  'data/GSI_ARMCO/initial_mag/spec_3.parquet',
  'data/GSI_ARMCO/initial_mag/spec_4.parquet'],
 'major_hyst_loop': ['data/GSI_ARMCO/major_hyst_loop/spec_1.parquet',
  'data/GSI_ARMCO/major_hyst_loop/spec_2.parquet',
  'data/GSI_ARMCO/major_hyst_loop/spec_3.parquet',
  'data/GSI_ARMCO/major_hyst_loop/spec_4.parquet'],
 'minor_hyst_loop': ['data/GSI_ARMCO/minor_hyst_loop/spec_1.parquet',
  'data/GSI_ARMCO/minor_hyst_loop/spec_2.parquet',
  'data/GSI_ARMCO/minor_hyst_loop/spec_3.parquet',
  'data/GSI_ARMCO/minor_hyst_loop/spec_4.parquet']}

In [3]:
val_data_paths = data_manifest["major_hyst_loop"][3:] + data_manifest["minor_hyst_loop"][2:]
train_data_paths = data_manifest["major_hyst_loop"][:3] + data_manifest["minor_hyst_loop"][:2]

# train_data_paths += data_manifest["initial_mag"]  # 40 points per specimen only

In [4]:
# check all dataframes H, B for max and min values

H_min, H_max = float("inf"), float("-inf")
B_min, B_max = float("inf"), float("-inf")
for path in train_data_paths + val_data_paths:
    df = pd.read_parquet(path)

    H_min = min(H_min, df["H"].min())
    H_max = max(H_max, df["H"].max())
    B_min = min(B_min, df["B"].min())
    B_max = max(B_max, df["B"].max())

print(f"H range: {H_min} to {H_max}")
print(f"B range: {B_min} to {B_max}")

H range: -22601.0 to 24191.0
B range: -2.0363 to 2.0505


In [5]:
datamodule = EncoderDecoderDataModule(
    known_covariates=["H"],
    target_covariate="B",
    train_df_paths=train_data_paths,
    val_df_paths=val_data_paths,
    ctxt_seq_len=10,
    tgt_seq_len=100,
    randomize_seq_len=False,
    min_ctxt_seq_len=5,
    min_tgt_seq_len=10,
    batch_size=16,
    normalize=False,  # important, add custom scalers below
    extra_transforms={
        "H": [MinMaxScaler(num_features_=1, min_=0, max_=1.0, data_min_=-22601, data_max_=24191)],
        "B": [MinMaxScaler(num_features_=1, min_=0, max_=1.0, data_min_=-2.0364, data_max_=2.0506)],
    },
    num_workers=2,
)
datamodule.prepare_data()
datamodule.setup()

/Users/antonlu/code/cern/transformertf/transformertf/data/datamodule/_base.py:921: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  torch.from_numpy(df[cov.col].to_numpy())


In [6]:
datamodule.transforms

{'H': TransformCollection(MinMaxScaler()),
 'B': TransformCollection(MinMaxScaler())}

In [7]:
module = EncoderDecoderPreisachNN(
    encoder=PreisachLSTMEncoder(
        hidden_dim=8,
        num_features=2,
        dropout=0.1,
        num_layers=1,
    ),
    hidden_dim=32,
    num_layers=2,  # MLP layers
    mesh_density_function=DefaultMeshSizeFunction(scale=0.5, offset=0.05),
    # mesh_density_function="exponential",
    mesh_scale=0.075,
    mesh_perturbation_std=0.05,
    m_scale_bounds=(0.0, 2.0),
    lr=1e-3,  # MLP
    lr_scale=1e-3,  # M, H scales
    lr_encoder=1e-4,  # encoder RNN
    normalized_density=True,
    temp=1e-1,
    temp_min=1e-3,
    temp_anneal_steps=10000,
    lr_step_interval=50,
    linear_fit_steps=0,
    encoder_fit_steps=2000,
    density_fit_steps=2000,
    fit_scale_offset=False,
    density_prior=CompositeDensityPrior(
        DiagonalDensityPrior(weight=0.1),
        SymmetryDensityPrior(weight=0.5),
        # CentroidDensityPrior(weight=0.01),  # since its a soft material
    ),
    # adaptive_loss_weights=True,
    # lr_adaptive=1e-4,
)

# print the number of parameters in the model
num_params = sum(p.numel() for p in module.parameters())
print(f"Number of parameters in the model: {num_params}")

print(f"Number of mesh points: {(module.model.base_mesh.numel() / 2)}")

# module.model.m_offset.raw_parameter.requires_grad_(False)

Number of parameters in the model: 2376
Number of mesh points: 6776.0


/var/folders/cs/zkc0wm4s4bv83yn4l6t2mdmw0000gn/T/ipykernel_20887/3844942184.py:29: UserWarning: SymmetryDensityPrior assumes mu(b,a) = mu(1-a,1-b), which is only valid for materials with symmetric major hysteresis loops.
  SymmetryDensityPrior(weight=0.5),


In [8]:
train_dataloader = datamodule.train_dataloader()
print(f"Number of batches in train dataloader: {len(train_dataloader)}, samples: {len(train_dataloader.dataset)}")

val_dataloader = datamodule.val_dataloader()[0]
print(f"Number of batches in val dataloader: {len(val_dataloader)}, samples: {len(val_dataloader.dataset)}")

Number of batches in train dataloader: 42, samples: 671
Number of batches in val dataloader: 3, samples: 3


In [9]:
batch = next(iter(train_dataloader))

module.common_step(batch, batch_idx=0)

/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  return torch.from_numpy(o.to_numpy(dtype=np.float64)).to(get_dtype(dtype))
/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the

{'loss': tensor(3.5014, grad_fn=<AddBackward0>),
 'aux_loss': tensor(1.7776),
 'physics_loss': tensor(1.9339),
 'saturation_reg': tensor(0.9934),
 'prior_loss': tensor(0.0468),
 'prior_losses': {'diagonal': tensor(0.0045), 'symmetry': tensor(0.0424)},
 'mse': tensor(0.1776),
 'rmse': tensor(0.4215),
 'mae': tensor(0.3291),
 'y_hat': tensor([[0.9816, 0.9819, 0.9821,  ..., 0.9911, 0.9914, 0.9917],
         [0.9816, 0.9818, 0.9818,  ..., 0.9933, 0.9937, 0.9940],
         [0.9788, 0.9789, 0.9789,  ..., 0.9903, 0.9906, 0.9909],
         ...,
         [0.9722, 0.9729, 0.9733,  ..., 0.5148, 0.4794, 0.4541],
         [0.9690, 0.9696, 0.9701,  ..., 0.3634, 0.3570, 0.3512],
         [0.2272, 0.1540, 0.1236,  ..., 0.7852, 0.7912, 0.7972]],
        grad_fn=<AddBackward0>),
 'y': tensor([[0.0278, 0.0288, 0.0299,  ..., 0.9508, 0.9521, 0.9534],
         [0.0312, 0.0324, 0.0335,  ..., 0.9591, 0.9603, 0.9615],
         [0.0299, 0.0310, 0.0321,  ..., 0.9534, 0.9546, 0.9558],
         ...,
         [0.67

In [10]:
trainer = L.Trainer(
    max_epochs=1000,
    accelerator="cpu",
    callbacks=[
        PlotHysteresisCallback(
            hysteron_scatter=True,
            train_plot_interval=50,
            validate_every_n_epochs=1,
            num_samples=5
        ),
        L.pytorch.callbacks.RichModelSummary(max_depth=2),
        L.pytorch.callbacks.LearningRateMonitor(logging_interval="epoch"),
        L.pytorch.callbacks.RichProgressBar(),
        L.pytorch.callbacks.EarlyStopping(monitor="validation/loss/dataloader_idx_0", patience=500, mode="min"),
        L.pytorch.callbacks.ModelCheckpoint(monitor="validation/loss/dataloader_idx_0", save_top_k=3, mode="min")
   ],
   min_epochs=10,
#    val_check_interval=1.0,
   log_every_n_steps=1,
   logger=L.pytorch.loggers.TensorBoardLogger(
       save_dir="logs",
       name="",
       default_hp_metric=False,
   )
    # logger=L.pytorch.loggers.WandbLogger(
    #     project="SA-PreisachNN"
    # )
)

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [11]:
trainer.fit(module, datamodule=datamodule)

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                     ┃ Type                          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model                    │ EncoderDecoderPreisachNNModel │  2.4 K │ train │     0 │
│ 1 │ model.encoder            │ PreisachLSTMEncoder           │  1.1 K │ train │     0 │
│ 2 │ model.density            │ ResNetMLP                     │  1.3 K │ train │     0 │
│ 3 │ model.density_activation │ Sigmoid                       │      0 │ train │     0 │
│ 4 │ model.h_scale            │ GPyConstrainedParameter       │      1 │ train │     0 │
│ 5 │ model.m_scale            │ GPyConstrainedParameter       │      1 │ train │     0 │
│ 6 │ model.m_offset           │ GPyConstrainedParameter       │      1 │ train │     0 │
│ 7 │ model.density_prior      │ CompositeDensityPrior         │      3 │ train │     0 │
└───┴──────────────────────────┴───────────────────────────────┴────────┴───────┴───────┘

Trainable params: 2.4 K                                                                                            
Non-trainable params: 3                                                                                            
Total params: 2.4 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 49                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  return torch.from_numpy(o.to_numpy(dtype=np.float64)).to(get_dtype(dtype))
/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the

/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  return torch.from_numpy(o.to_numpy(dtype=np.float64)).to(get_dtype(dtype))
/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the

/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  return torch.from_numpy(o.to_numpy(dtype=np.float64)).to(get_dtype(dtype))
/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the

/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  return torch.from_numpy(o.to_numpy(dtype=np.float64)).to(get_dtype(dtype))
/Users/antonlu/code/cern/transformertf/transformertf/data/_dtype.py:53: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the

In [12]:
module.model.m_scale.value

tensor(0.5000)

The full physical relation is B = μ₀(H + M). The h_scale term is meant to capture the μ₀H part. Setting it to zero is physically justified for high-permeability soft magnets (like ARMCO iron) where μᵣ ≫ 1, meaning μ₀H ≪ μ₀M ≈ B — the direct field contribution is negligible compared to the magnetization term.

So it's not exactly B = M, it's more precisely B ≈ μ₀M, with the normalization absorbing the μ₀ factor. Whether this approximation holds depends on how far into the linear (pre-saturation) region your data reaches — if you have significant data near H turning points where M hasn't caught up, you might actually need a nonzero h_scale.



* do initial state activations have the correct sign? plots would suggest the opposite
* should we actually do $B=H+\mu M$, which needs $H$ and $M$ to have the correct scale. now we have `h_scale`, `m_scale`, and `m_offset`.
* In case of yes, should we learn the scales at all? when?
* Do the learning stages make sense, because initial states determine where magen